# Optuna (hyperparameter optimization)

**Domain:** AI/ML Tooling  ·  **recommended addition**  ·  **runnable:** yes

A define-by-run framework for hyperparameter optimization: you write a normal Python
function, sample hyperparameters inside it with `trial.suggest_*`, return a score, and
Optuna searches the space for you with smart sampling and early stopping.

## 1. What & Why

**What it is.** Optuna is a black-box optimization library. You give it an *objective
function* that takes a `trial`, samples hyperparameters from it, trains/evaluates a model,
and returns a number. Optuna repeatedly calls that function (each call is a **trial**),
choosing the next set of hyperparameters based on what it has seen so far, and tracks the
best result in a **study**.

**The problem it solves.** Grid search explodes combinatorially and wastes budget on bad
regions; random search is better but blind. Manual tuning is slow and unreproducible.
Optuna gives you:

- **Sample-efficient search** — a Bayesian-style sampler (TPE) that concentrates trials
  where good values were found, instead of sweeping a fixed grid.
- **Pruning** — kill unpromising trials early (e.g. a model whose validation loss is awful
  after 2 epochs) so budget goes to candidates that matter.
- **Define-by-run search spaces** — the space is built *as the code runs*, so it can be
  dynamic and conditional (e.g. only suggest `n_layers` deep params when `model == "mlp"`).
- **Persistence & parallelism** — back a study with a database and many workers append to
  the same study concurrently.

**Reach for it when** you have more than a couple of hyperparameters, each evaluation is
non-trivial (model training, a simulation), and you want to spend a fixed budget wisely.
**Skip it when** you have one or two knobs and three candidate values each — just loop.

## 2. Mental Model

Think of Optuna as a **smart `for` loop with memory**.

```
study = create_study()                # the experiment + history of all trials
for _ in range(n_trials):
    trial  = study.ask()              # sampler proposes hyperparameters
    score  = objective(trial)         # YOU train & evaluate with them
    study.tell(trial, score)          # result feeds back into the sampler
best = study.best_params              # the winner
```

`study.optimize(objective, n_trials=N)` is just sugar over that ask/tell loop. Two pieces
do the real work:

- The **sampler** decides *what to try next* (TPE by default — it models which
  hyperparameter values tend to produce good vs. bad scores and samples from the "good"
  distribution).
- The **pruner** decides *when to give up* on the trial currently running, by watching
  intermediate values you report during training.

Everything else (storage, visualization, distributed workers) hangs off the `study`
object holding the trial history.

## 3. Key Concepts

| Term | What it is |
|------|------------|
| **Study** | The optimization session: search direction, sampler, pruner, and the full trial history. Create with `optuna.create_study(direction=...)`. |
| **Trial** | One evaluation of the objective. Inside it you call `trial.suggest_*` to draw hyperparameters; it also records intermediate values for pruning. |
| **Objective function** | `def objective(trial) -> float`. Returns the metric to optimize. Can return a tuple for multi-objective. |
| **`suggest_*`** | How you define the space, define-by-run: `suggest_float(name, lo, hi, log=True)`, `suggest_int`, `suggest_categorical(name, [...])`. The *name* is the key Optuna tracks. |
| **Sampler** | Strategy for proposing the next trial. `TPESampler` (default), `RandomSampler`, `GridSampler`, `CmaEsSampler`, `QMCSampler`. |
| **Pruner** | Early-stopping for trials. `MedianPruner` (default), `SuccessiveHalvingPruner`, `HyperbandPruner`. Driven by `trial.report(value, step)` + `trial.should_prune()`. |
| **`direction`** | `"minimize"` (loss) or `"maximize"` (accuracy). Multi-objective uses `directions=[...]`. |
| **Storage** | Where trials live. In-memory by default; pass `storage="sqlite:///study.db"` to persist and to let multiple processes share one study. |
| **`study.best_value` / `best_params` / `best_trial`** | The winning score, its hyperparameters, and the full trial object. |

## 4. Setup

Pure Python, CPU-friendly. The visualization helpers need `plotly` (and dashboards use the
separate `optuna-dashboard` package), but the core optimizer has no heavy dependencies.

```bash
%pip install optuna scikit-learn        # core + the dataset/model used below
# %pip install optuna plotly             # for optuna.visualization.*
# %pip install optuna-dashboard          # live web UI over a stored study
```

In [1]:
import optuna
import sklearn

# Optuna logs every trial at INFO by default; quiet it so output stays readable.
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("optuna", optuna.__version__)
print("scikit-learn", sklearn.__version__)

/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


optuna 4.9.0
scikit-learn 1.9.0


## 5. Worked Examples

### Example 1 — the anatomy of a study (minimize a function)

Before any ML, optimize a plain math function so the moving parts are obvious. We minimize
`(x - 2)^2 + (y + 3)^2`, whose true optimum is `x=2, y=-3` with value `0`. The objective
*samples* `x` and `y` from the trial and *returns* the value to minimize.

In [2]:
def objective(trial):
    x = trial.suggest_float("x", -10.0, 10.0)
    y = trial.suggest_float("y", -10.0, 10.0)
    return (x - 2) ** 2 + (y + 3) ** 2

# A fixed seed makes the TPE sampler reproducible for this demo.
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=100)

print(f"best value : {study.best_value:.5f}")
print(f"best params: {study.best_params}")
print(f"n trials   : {len(study.trials)}")

best value : 0.02505
best params: {'x': 1.939661722963459, 'y': -3.146322271946312}
n trials   : 100


TPE homes in on `x≈2, y≈-3` within 100 cheap trials. Note we never enumerated a grid —
the search space was declared *inside* the objective by the two `suggest_float` calls.

### Example 2 — tuning a real model, with pruning

Now tune a `RandomForestClassifier` on the small built-in *wine* dataset (178 rows, CPU in
under a second). Two things to notice:

1. **Conditional / mixed space** — integers, a log-scale float, and a categorical, all
   declared define-by-run.
2. **Pruning** — we score each candidate via incremental cross-validation folds and
   `trial.report` the running mean after each fold; `MedianPruner` aborts trials that are
   already tracking below the median of completed trials, so wasted compute is reclaimed.

In [3]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

X, y = load_wine(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400, step=50),
        "max_depth": trial.suggest_int("max_depth", 2, 16),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-4, 1e-1, log=True),
        "random_state": 0,
    }
    clf = RandomForestClassifier(**params)

    # Score fold-by-fold so we can report intermediate values and let the pruner act.
    scores = []
    for step, (tr, va) in enumerate(cv.split(X, y)):
        clf.fit(X[tr], y[tr])
        scores.append(clf.score(X[va], y[va]))
        trial.report(float(np.mean(scores)), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(scores))


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study.optimize(objective, n_trials=40)

states = [t.state.name for t in study.trials]
print(f"best CV accuracy: {study.best_value:.4f}")
print(f"best params     : {study.best_params}")
print(f"completed / pruned: {states.count('COMPLETE')} / {states.count('PRUNED')}")

best CV accuracy: 0.9832
best params     : {'n_estimators': 350, 'max_depth': 6, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'ccp_alpha': 0.0030586566669785274}
completed / pruned: 35 / 5


The pruner typically discards a chunk of the 40 trials before all 5 folds run, and the
TPE sampler still finds a strong configuration. A few more handles worth knowing:

- **Inspect as a table:** `study.trials_dataframe()` returns a pandas frame of every trial
  (params, value, state, duration) — great for ad-hoc analysis.
- **Visualize** (needs `plotly`): `optuna.visualization.plot_optimization_history(study)`,
  `plot_param_importances(study)`, `plot_contour(study)`.
- **Persist & resume:** add `storage="sqlite:///wine.db"` and
  `study_name="rf", load_if_exists=True` to `create_study`; rerun or launch more workers
  and they extend the same study.

## 6. Gotchas & Pitfalls

- **`suggest_*` names must be stable and unique.** Optuna keys the search space by the
  string name. Reusing one name for two different parameters, or building names in a loop
  without distinct suffixes, silently corrupts the space.
- **Direction must match the metric.** Returning a *loss* under `direction="maximize"` (or
  accuracy under `"minimize"`) optimizes the wrong way — a classic silent bug.
- **Pruning needs honest intermediate values.** `MedianPruner` only works if you call
  `trial.report(value, step)` with a metric that improves monotonically-ish over steps
  (epochs, folds, boosting rounds) *and* check `trial.should_prune()`. Reporting noise, or
  forgetting the `should_prune()` check, makes pruning useless or harmful.
- **TPE is sequential by design.** It's most sample-efficient run serially; massively
  parallel workers see staler history. For heavy parallelism consider `RandomSampler` or
  `QMCSampler`, or accept slightly less efficient suggestions.
- **Seeding ≠ full reproducibility.** Seeding the sampler fixes *its* proposals, but your
  model, data splits, and any GPU nondeterminism still vary unless you seed those too.
- **In-memory studies vanish on crash.** For anything long-running, use a `storage` URL so
  trials survive process death and can resume.
- **Log scale for multiplicative ranges.** Learning rates, regularization, etc. span orders
  of magnitude — use `suggest_float(..., log=True)`, not a linear range, or the sampler
  wastes trials in the large-value tail.
- **Categorical choices are unordered.** `suggest_categorical` treats options as a set with
  no notion of distance; don't smuggle a numeric range through it.

## 7. When to Use vs Alternatives

| Option | Strengths | Weaknesses / when not to |
|--------|-----------|--------------------------|
| **Optuna** | Define-by-run dynamic spaces, strong TPE + pruning, easy persistence/parallelism, framework-agnostic, good viz. | Pure single-machine TPE isn't ideal for thousand-worker fleets; not a full experiment tracker. |
| **`sklearn` GridSearchCV** | Dead simple, exhaustive, reproducible. | Combinatorial blowup; no pruning; no learning between candidates. Fine for ≤2–3 small knobs. |
| **`sklearn` RandomizedSearchCV** | Better budget use than grid, trivial API. | Blind — never exploits past results; no pruning. |
| **Ray Tune** | Built for large distributed/parallel HPO; many schedulers (ASHA, PBT); *can use Optuna as its search algorithm*. | Heavier infra and concepts; overkill on one machine. |
| **Hyperopt** | Early TPE library, similar spirit. | Less active, weaker tooling/viz, no pruning story as clean as Optuna's. |
| **Weights & Biases Sweeps** | Tight integration with W&B tracking/dashboards. | Couples you to W&B; sampling/pruning less flexible than Optuna. |

**Rule of thumb:** a few discrete knobs → just loop or `GridSearchCV`. Real search on one
or a few machines → **Optuna**. Cluster-scale parallel HPO → **Ray Tune** (often *with*
Optuna as the underlying sampler).

## 8. Resources

- **Official docs** — https://optuna.readthedocs.io/en/stable/
- **Key features tutorial** (efficient optimization, pruning, viz) — https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/index.html
- **GitHub repo & examples** (PyTorch, LightGBM, Keras, etc.) — https://github.com/optuna/optuna-examples
- **Samplers & pruners reference** — https://optuna.readthedocs.io/en/stable/reference/samplers/index.html
- **Optuna Dashboard** (live web UI over a stored study) — https://github.com/optuna/optuna-dashboard
- **Paper:** Akiba et al., *Optuna: A Next-generation Hyperparameter Optimization Framework* (KDD 2019) — https://arxiv.org/abs/1907.10902